# Fitness Optimisation

This notebook is the analysis/front-end companion to the updated optimisation backend in `fitness_backend.py`. It uses the same goal set and output schema as the Flask API.

Supported goals:
- `weight_loss`
- `muscle_gain`
- `body_recomposition`

`body_recomposition` keeps calories at maintenance, pushes protein higher, and prioritises resistance training over cardio.

In [2]:
# install pandas
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 81.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 72.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install gurobipy 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 63.8 MB/s  0:00:006m0:00:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / 'AM13 - Decision Analytics & Modelling' / 'Group Project',
]
project_dir = next((path for path in candidate_dirs if (path / 'fitness_backend.py').exists()), None)
if project_dir is None:
    raise FileNotFoundError('Could not locate fitness_backend.py from the current working directory.')
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from fitness_backend import (
    VALID_GOALS,
    build_default_profile,
    get_model_counts,
    run_fitopt_pipeline,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
print(f'Using backend module from: {project_dir}')
print(f'Valid goals: {VALID_GOALS}')

Using backend module from: /workspaces/optifit
Valid goals: ('weight_loss', 'muscle_gain', 'body_recomposition')


## Model Size

Exact Gurobi counts can differ slightly by goal. `body_recomposition` has two additional workout constraints because it enforces a resistance floor and cardio cap.

In [6]:
counts_by_goal = {}
for goal in VALID_GOALS:
    profile = build_default_profile()
    profile['goal'] = goal
    counts_by_goal[goal] = get_model_counts(profile)

counts_df = pd.DataFrame(counts_by_goal).T
counts_df

Restricted license - for non-production use only - expires 2027-11-29


,nutrition_variables,nutrition_constraints,workout_variables,workout_constraints,total_variables,total_constraints
weight_loss,364,70,28,25,392,95
muscle_gain,364,70,28,25,392,95
body_recomposition,364,70,28,27,392,97


## Configure a Scenario

Edit the profile below and rerun the remaining cells. The payload returned by `run_fitopt_pipeline()` matches the Flask API response structure.

In [7]:
profile = build_default_profile()


profile.update({
    'goal': 'body_recomposition',
    'fitness_level': 'intermediate',
    'age': 26,
    'sex': 'male',
    'height_cm': 182,
    'weight_kg': 80.0,
    'target_kg': 78.0,
    'budget_day': 10.0,
    'gym_days': 4,
    'time_per_session': 1.25,
    'equipment_access': 'full_gym',
    'dietary_restrictions': [],
    'food_preferences_enjoy': ['lean_protein', 'grains', 'fruit'],
    'food_preferences_avoid': ['fish'],
    'availability': {
        'monday': 1,
        'tuesday': 1,
        'wednesday': 1,
        'thursday': 0,
        'friday': 1,
        'saturday': 1,
        'sunday': 0,
    },
})

profile

{'goal': 'body_recomposition',
 'fitness_level': 'intermediate',
 'sex': 'male',
 'age': 26,
 'height_cm': 182,
 'weight_kg': 80.0,
 'target_kg': 78.0,
 'weeks': 8,
 'budget_day': 10.0,
 'gym_days': 4,
 'time_per_session': 1.25,
 'use_macro_override': False,
 'macro_targets': {'protein_g': None, 'carbs_g': None, 'fat_g': None},
 'dietary_restrictions': [],
 'equipment_access': 'full_gym',
 'food_preferences_enjoy': ['lean_protein', 'grains', 'fruit'],
 'food_preferences_avoid': ['fish'],
 'availability': {'monday': 1,
  'tuesday': 1,
  'wednesday': 1,
  'thursday': 0,
  'friday': 1,
  'saturday': 1,
  'sunday': 0}}

In [8]:
result = run_fitopt_pipeline(profile)
list(result.keys())

['summary', 'projection', 'meal_plan', 'workout_schedule', 'insights']

## Summary

In [9]:
summary_df = pd.DataFrame([result['summary']])
summary_df

,weeks_to_goal,daily_calories,weekly_gym_hours,weekly_food_cost
0,11.16,2768.8,4.0,69.98


## Projection

In [10]:
projection_df = pd.DataFrame(result['projection'])
projection_df

,week,weight_kg
0,1,79.82
1,2,79.64
2,3,79.46
3,4,79.28
4,5,79.10
5,6,78.92
6,7,78.75
7,8,78.57
8,9,78.39
9,10,78.21


## Meal Plan

In [11]:
meal_rows = []
for day, entries in result['meal_plan'].items():
    for entry in entries:
        meal_rows.append({'day': day, **entry})

meal_plan_df = pd.DataFrame(meal_rows)
meal_plan_df

,day,meal,food,grams,calories,protein,carbs,fat
0,monday,Breakfast,Oats,298.9,1162.8,50.8,197.3,20.9
1,monday,Dinner,Chicken Breast,469.9,775.3,145.7,0.0,16.9
2,monday,Lunch,Oats,177.9,692.2,30.2,117.4,12.5
3,monday,Snack,Oats,26.9,104.7,4.6,17.8,1.9
4,monday,Snack,Olive Oil,3.8,33.8,0.0,0.0,3.8
5,tuesday,Breakfast,Oats,156.6,609.0,26.6,103.3,11.0
6,tuesday,Dinner,Oats,177.9,692.2,30.2,117.4,12.5
7,tuesday,Lunch,Oats,169.3,658.4,28.8,111.7,11.8
8,tuesday,Lunch,Olive Oil,3.8,33.8,0.0,0.0,3.8
9,tuesday,Snack,Chicken Breast,469.9,775.3,145.7,0.0,16.9


## Workout Schedule

In [12]:
workout_rows = []
for day, entries in result['workout_schedule'].items():
    if not entries:
        workout_rows.append({
            'day': day,
            'exercise': 'Rest day',
            'sets': 0,
            'reps': 0,
            'duration_min': 0,
            'muscle_group': 'rest',
        })
        continue
    for entry in entries:
        workout_rows.append({'day': day, **entry})

workout_schedule_df = pd.DataFrame(workout_rows)
workout_schedule_df

,day,exercise,sets,reps,duration_min,muscle_group
0,monday,Rest day,0,0,0,rest
1,tuesday,Barbell Squat,4,10,60,lower_body
2,wednesday,Push Pull Upper,4,10,60,upper_body
3,thursday,Rest day,0,0,0,rest
4,friday,Push Pull Upper,4,10,60,upper_body
5,saturday,Barbell Squat,4,10,60,lower_body
6,sunday,Rest day,0,0,0,rest


## Sensitivity Insights

In [13]:
insights_df = pd.DataFrame(result['insights'])
insights_df

,text
0,Adding one more gym day shifts weekly gym time...
1,The plan uses 10.0 of the 10.0 daily food budg...
2,Body recomposition keeps calories at maintenan...


## API-Ready JSON Preview

In [14]:
print(json.dumps(result, indent=2))

{
  "summary": {
    "weeks_to_goal": 11.16,
    "daily_calories": 2768.8,
    "weekly_gym_hours": 4.0,
    "weekly_food_cost": 69.98
  },
  "projection": [
    {
      "week": 1,
      "weight_kg": 79.82
    },
    {
      "week": 2,
      "weight_kg": 79.64
    },
    {
      "week": 3,
      "weight_kg": 79.46
    },
    {
      "week": 4,
      "weight_kg": 79.28
    },
    {
      "week": 5,
      "weight_kg": 79.1
    },
    {
      "week": 6,
      "weight_kg": 78.92
    },
    {
      "week": 7,
      "weight_kg": 78.75
    },
    {
      "week": 8,
      "weight_kg": 78.57
    },
    {
      "week": 9,
      "weight_kg": 78.39
    },
    {
      "week": 10,
      "weight_kg": 78.21
    },
    {
      "week": 11,
      "weight_kg": 78.03
    },
    {
      "week": 12,
      "weight_kg": 78.0
    }
  ],
  "meal_plan": {
    "monday": [
      {
        "meal": "Breakfast",
        "food": "Oats",
        "grams": 298.9,
        "calories": 1162.8,
        "protein": 50.8,
       